# Bulk Video Creator - Notebook (Sequential Execution)

This notebook executes the bulk operations functionality in two sequential phases:
- **Phase 1**: Sheet Upload & GCS Organization
- **Phase 2**: Video Generation from Bulk Data

Execute cells sequentially to process your bulk video generation tasks.

## Section 1: Setup and Installation

In [ ]:
# Install Required Dependencies
import subprocess
import sys

packages = [
    'google-auth',
    'google-auth-oauthlib',
    'google-auth-httplib2',
    'google-api-python-client',
    'google-cloud-storage',
    'google-genai',
    'requests',
    'python-dateutil'
]

for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '-q', package])
    except Exception as e:
        print(f"Warning: Could not install {package}: {e}")

print("✓ Dependencies installed successfully!")
print("\nIMPORTANT: If running in Google Colab, please restart the runtime (Runtime > Restart session) now so Python loads the newly upgraded packages.")

In [ ]:
# Import Required Libraries
import os
import json
import logging
import datetime
import time
import re
import functools
from typing import Any, Optional
from io import BytesIO
from concurrent import futures
import requests
from dataclasses import dataclass
import base64

import google.auth
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from google.cloud import storage
import vertexai
from vertexai.generative_models import GenerativeModel, Part, Image

# Setup Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('BulkVideoCreator')

print("✓ All imports loaded successfully!")

In [ ]:
# ============================================================================
# Google Colab Authentication (Run this FIRST if using Colab)
# ============================================================================

import sys

colab_available = False
try:
    from google.colab import auth
    colab_available = True
except ImportError:
    pass

if colab_available:
    try:
        print("📌 Running in Google Colab - Authenticating with your Google account...")
        print("⏳ Please wait for the authentication popup to appear...")
        print("   (You may need to click 'Authorize' when prompted)\n")
        
        # Authenticate with a timeout
        auth.authenticate_user()
        print("✓ Colab authentication successful!\n")
        
    except Exception as e:
        print(f"⚠️  Authentication popup may have been skipped or timed out: {e}")
        print("   This is OK - we'll try again when needed.\n")
else:
    print("ℹ️  Not running in Google Colab.")
    print("   For local use, ensure: gcloud auth application-default login\n")

## Section 2: Configuration Variables

Configure all parameters below for your bulk operations:

In [ ]:
# ============================================================================
# CONFIGURATION VARIABLES - Modify these for your execution
# ============================================================================

# Google Cloud Configuration
# Values are read from environment variables (same names as the backend
# .env-template). Set them before starting Jupyter, or edit the placeholders.
import os
GCP_PROJECT_ID = os.getenv("PROJECT_ID", "your-project-id")
GCS_BUCKET = os.getenv("GCS_BUCKET", "your-bucket-name")
LOCATION = os.getenv("LOCATION", "us-central1")
BULK_GCS_PREFIX = os.getenv("BULK_GCS_PREFIX", "bulk-video-creator")

# Video Generation Model
VIDEO_MODEL = os.getenv("BULK_VIDEO_MODEL", "veo-3.1-generate-001")

# Google Sheets Configuration
SHEET_URL = os.getenv(
    "SHEET_URL", "https://docs.google.com/spreadsheets/d/YOUR_SHEET_ID/edit"
)

# Expected Sheet Columns (required in your spreadsheet)
# - image_url: URL of the image to download
# - image_metadata: JSON metadata associated with the image
# - video_output: Output path identifier for video generation
# - status: Status column (will be overwritten)
# - logs: Logs column (will be overwritten)

# Video Generation Parameters
VIDEO_GENERATION_PROMPT = os.getenv(
    "BULK_VIDEO_PROMPT",
    "Animate the static photograph with extremely subtle, cinematic motion. The feeling is **serene** and **aspirational**. Use a very **slow, gentle camera zoom (push-in)**, a **subtle parallax shift** between foreground and background, and a **soft, atmospheric breathing** effect. The overall movement must be minimal and dreamy, like a high-end film still coming to life.",
)

VIDEO_DURATION_SECONDS = int(os.getenv("BULK_VIDEO_DURATION_SECS", "4"))  # 4, 6 or 8
VIDEO_QUALITY = "HIGH"              # Video quality (LOW, MEDIUM, HIGH)
MAX_PARALLEL_WORKERS = 3            # Max parallel video generation tasks
VIDEO_FPS = 24                      # Frames per second

# Processing Options
SKIP_ROWS_WITHOUT_IMAGE = True      # Skip rows where image_url is empty
CONTINUE_ON_ERROR = True            # Continue processing if a row fails
VERBOSE_LOGGING = True              # Enable detailed logging

# Phase Tracking
PHASE_1_COMPLETE = False            # Set to True after Phase 1 completes
BULK_ID_FROM_PHASE_1 = None         # Will be populated after Phase 1
PHASE_1_STATUS_SUMMARY = {}         # Status summary from Phase 1

print("✓ Configuration variables initialized")
print(f"  Project: {GCP_PROJECT_ID}")
print(f"  Bucket: {GCS_BUCKET}")
print(f"  Sheet: {SHEET_URL}")
print(f"  Region: {LOCATION}")
print(f"  Video Model: {VIDEO_MODEL}")

In [ ]:
# ============================================================================
# Authenticate with Google Cloud (Credentials should already be set up)
# ============================================================================

def setup_gcp_authentication(retry_count=0):
    """Initialize GCP services using authenticated credentials"""
    try:
        print(f"Attempting to get GCP credentials (attempt {retry_count + 1})...")
        
        # Get default credentials (already authenticated from Colab cell)
        credentials, project = google.auth.default(
            scopes=[
                'https://www.googleapis.com/auth/spreadsheets',
                'https://www.googleapis.com/auth/cloud-platform'
            ]
        )
        
        logger.info("✓ Credentials obtained successfully")
        
        # Initialize services
        sheets_service = build('sheets', 'v4', credentials=credentials)
        storage_client = storage.Client(project=GCP_PROJECT_ID, credentials=credentials)
        
        logger.info("✓ GCP services initialized")
        print(f"✓ GCP Authentication successful!")
        return credentials, sheets_service, storage_client
        
    except Exception as e:
        logger.error(f"Authentication attempt {retry_count + 1} failed: {e}")
        
        # If in Colab and auth failed, try again with explicit authentication
        if retry_count == 0:
            try:
                from google.colab import auth
                print("\n⚠️  Retrying with explicit Colab authentication...")
                auth.authenticate_user()
                print("✓ Colab re-authenticated. Trying again...\n")
                return setup_gcp_authentication(retry_count + 1)
            except Exception as colab_error:
                logger.warning(f"Colab auth retry failed: {colab_error}")
        
        # If we've retried, give up with helpful message
        raise RuntimeError(
            "Could not authenticate with Google Cloud.\n\n"
            "Please try one of these solutions:\n"
            "1. COLAB USERS: Run this cell again (credentials may have timed out)\n"
            "2. COLAB USERS: Clear output (Edit → Clear all outputs) and try again\n"
            "3. LOCAL USERS: Run in terminal: gcloud auth application-default login\n"
            f"\nError: {e}"
        )

# Run authentication
try:
    credentials, sheets_service, storage_client = setup_gcp_authentication()
    gcs_bucket = storage_client.bucket(GCS_BUCKET)
    logger.info(f"✓ Connected to GCS bucket: {GCS_BUCKET}")
    print(f"✓ All GCP services ready!")
    print(f"  - Project: {GCP_PROJECT_ID}")
    print(f"  - Bucket: {GCS_BUCKET}\n")
    
except Exception as e:
    logger.error(f"GCP setup failed: {e}")
    print("\n" + "=" * 80)
    print("❌ GCP AUTHENTICATION FAILED")
    print("=" * 80)
    print(f"\n{e}\n")
    raise

In [ ]:
# ============================================================================
# Utility Functions and Helper Classes
# ============================================================================

def get_bulk_bucket_base_path(bulk_id: str) -> str:
    """Construct GCS path for bulk operations"""
    return f"gs://{GCS_BUCKET}/{BULK_GCS_PREFIX}/{bulk_id}/"

def get_bulk_row_image_bucket_base_path(bulk_id: str, row_id: str) -> str:
    """Construct GCS path for row seed images"""
    return f"{get_bulk_bucket_base_path(bulk_id)}{row_id}/images_seeds/"

def get_bulk_row_video_output_base_path(bulk_id: str, row_id: str) -> str:
    """Construct GCS path for row video output"""
    return f"{get_bulk_bucket_base_path(bulk_id)}{row_id}/video_output/"

def get_segment_number_from_row_id(row_id: str) -> int:
    """Extract numeric part from row_id (e.g., 'row_1' -> 1)"""
    try:
        return int(row_id.split('_')[-1])
    except (ValueError, IndexError) as e:
        raise ValueError(f"Invalid row_id format: {row_id}") from e

def get_file_name_from_uri(uri: str) -> str:
    """Extract filename from URI"""
    return uri.split('/')[-1]

def get_mime_type_from_filename(filename: str) -> str:
    """Get MIME type based on file extension"""
    ext = filename.lower().split('.')[-1]
    mime_types = {
        'jpg': 'image/jpeg',
        'jpeg': 'image/jpeg',
        'png': 'image/png',
        'gif': 'image/gif',
        'webp': 'image/webp'
    }
    return mime_types.get(ext, 'image/jpeg')

def get_signed_uri_from_gcs_uri(gcs_uri: str) -> str:
    """Generate signed URL from GCS URI (valid for 1 hour)"""
    try:
        if gcs_uri.startswith("gs://"):
            _, path = gcs_uri.split("gs://", 1)
            bucket_name, blob_name = path.split("/", 1)
            
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            
            # Generate signed URL
            signed_url = blob.generate_signed_url(
                version="v4",
                expiration=datetime.timedelta(hours=1),
                method="GET"
            )
            return signed_url
    except Exception as e:
        logger.warning(f"Could not generate signed URL: {e}")
    return ""

@dataclass
class BulkVideoRequest:
    """Represents a single bulk video generation request"""
    bulk_id: str
    row_id: str
    prompt: str
    seed_image_uri: Optional[str] = None

print("✓ Utility functions loaded")

In [ ]:
# ============================================================================
# Google Sheets Service
# ============================================================================

class GoogleSheetsService:
    """Service to interact with Google Sheets API"""
    
    def __init__(self, sheets_service):
        self.sheets_service = sheets_service
        logger.info("GoogleSheetsService initialized")
    
    def extract_sheet_id_from_url(self, sheet_url: str) -> str:
        """Extract Google Sheet ID from URL"""
        # If already just an ID, return it
        if not sheet_url.startswith('http'):
            return sheet_url
        
        # Extract ID from URL
        match = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', sheet_url)
        if match:
            return match.group(1)
        
        raise ValueError(f"Could not extract sheet ID from URL: {sheet_url}")
    
    def get_sheet_data(self, sheet_url: str) -> list[dict[str, Any]]:
        """Fetch data from Google Sheet and convert to list of dicts"""
        try:
            sheet_id = self.extract_sheet_id_from_url(sheet_url)
            logger.info(f"Extracted sheet ID: {sheet_id}")
            
            # Get first sheet metadata
            spreadsheet = self.sheets_service.spreadsheets().get(
                spreadsheetId=sheet_id
            ).execute()
            
            if not spreadsheet.get('sheets'):
                raise ValueError("No sheets found in spreadsheet")
            
            first_sheet = spreadsheet['sheets'][0]
            sheet_title = first_sheet['properties']['title']
            range_name = f"{sheet_title}!A:Z"
            logger.info(f"Using range: {range_name}")
            
            # Fetch sheet data
            result = self.sheets_service.spreadsheets().values().get(
                spreadsheetId=sheet_id,
                range=range_name,
                valueRenderOption='FORMATTED_VALUE'
            ).execute()
            
            values = result.get('values', [])
            logger.info(f"Retrieved {len(values)} rows from sheet")
            
            if not values:
                logger.warning("No data found in sheet")
                return []
            
            # Convert to list of dicts
            headers = values[0]
            rows_data = []
            
            for row_index, row in enumerate(values[1:], start=1):
                # Pad row with empty strings if needed
                while len(row) < len(headers):
                    row.append('')
                
                row_dict = dict(zip(headers, row))
                rows_data.append(row_dict)
            
            logger.info(f"Converted {len(rows_data)} rows to dictionaries")
            return rows_data
            
        except Exception as e:
            logger.error(f"Error fetching sheet data: {e}")
            raise
    
    def get_sheet_title(self, sheet_id: str) -> str:
        """Get the title of the first sheet in a spreadsheet"""
        try:
            spreadsheet = self.sheets_service.spreadsheets().get(
                spreadsheetId=sheet_id
            ).execute()
            
            if not spreadsheet.get('sheets'):
                raise ValueError("No sheets found in spreadsheet")
            
            first_sheet = spreadsheet['sheets'][0]
            return first_sheet['properties']['title']
        except Exception as e:
            logger.error(f"Error getting sheet title: {e}")
            raise
    
    def batch_append_values(self, sheet_id: str, row_updates: list[list]) -> None:
        """
        Batch append/update rows in Google Sheet.
        
        Args:
            sheet_id: Google Sheet ID
            row_updates: List of [row_index, status, logs, video_output] values
        """
        try:
            if not row_updates:
                return
            
            sheet_title = self.get_sheet_title(sheet_id)
            
            # Build update requests for each row
            # Assuming columns: A=image_url, B=image_metadata, C=video_output, D=status, E=logs
            for row_index, status, logs, video_output in row_updates:
                row_num = row_index + 1  # Convert to 1-indexed (row 1 = headers)
                range_name = f"{sheet_title}!C{row_num}:E{row_num}"
                
                values = [[video_output, status, logs]]
                
                body = {'values': values}
                
                self.sheets_service.spreadsheets().values().update(
                    spreadsheetId=sheet_id,
                    range=range_name,
                    valueInputOption='RAW',
                    body=body
                ).execute()
                
                logger.info(f"Updated sheet row {row_num}: status={status}")
        
        except Exception as e:
            logger.error(f"Error in batch_append_values: {e}")
            # Don't raise - sheet update is optional for core functionality

# Initialize Google Sheets Service
sheets_svc = GoogleSheetsService(sheets_service)
print("✓ Google Sheets Service initialized")

## Section 3: PHASE 1 - Sheet Upload & GCS Organization

This phase:
1. Parses your Google Sheet
2. Downloads images from provided URLs
3. Organizes data into GCS bulk folder structure
4. Generates a bulk_id for tracking

**Expected Sheet Columns:** image_url, image_metadata, video_output, status, logs

In [ ]:
# ============================================================================
# PHASE 1: Sheet to Bulk Handler
# ============================================================================

class SheetToBulkHandler:
    """Handles conversion of Google Sheets data to GCS bulk structure"""
    
    def __init__(self, storage_client, sheets_service, bucket_name):
        self.storage_client = storage_client
        self.sheets_service = sheets_service
        self.bucket_name = bucket_name
        self.bucket = storage_client.bucket(bucket_name)
        logger.info(f"SheetToBulkHandler initialized for bucket: {bucket_name}")
    
    def generate_bulk_id(self) -> str:
        """Generate a unique bulk ID based on timestamp and sheet ID"""
        timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
        return f"{timestamp}"
    
    def _download_image_from_url(self, url: str) -> bytes:
        """Download image from URL"""
        try:
            logger.info(f"Downloading image from: {url}")
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            return response.content
        except Exception as e:
            logger.error(f"Failed to download image from {url}: {e}")
            raise
    
    def _upload_image_to_gcs(self, file_path: str, image_data: bytes, 
                             filename: str) -> str:
        """Upload image data to GCS and return public URL"""
        try:
            # Extract blob name from GCS URI (remove gs://bucket/ prefix)
            blob_name = file_path.replace(f"gs://{self.bucket_name}/", "")
            
            blob = self.bucket.blob(blob_name)
            mime_type = get_mime_type_from_filename(filename)
            blob.upload_from_string(image_data, content_type=mime_type)
            public_url = blob.public_url
            logger.info(f"Uploaded image to GCS: {blob_name}")
            return public_url
        except Exception as e:
            logger.error(f"Failed to upload image to GCS: {e}")
            raise
    
    def _upload_metadata_to_gcs(self, metadata_path: str, 
                                metadata_dict: dict) -> str:
        """Upload metadata JSON to GCS"""
        try:
            # Extract blob name from GCS URI (remove gs://bucket/ prefix)
            blob_name = metadata_path.replace(f"gs://{self.bucket_name}/", "")
            
            blob = self.bucket.blob(blob_name)
            blob.upload_from_string(
                json.dumps(metadata_dict, indent=2),
                content_type='application/json'
            )
            logger.info(f"Uploaded metadata to: {blob_name}")
            return metadata_path
        except Exception as e:
            logger.error(f"Failed to upload metadata: {e}")
            raise
    
    def _update_sheet_row_status(self, sheet_id: str, sheet_title: str, 
                                  row_idx: int, status: str, logs: str) -> None:
        """Update status and logs columns in the Google Sheet"""
        try:
            # Find the row index + 1 (Google Sheets is 1-indexed, and row 1 is headers)
            row_num = row_idx + 1
            
            # Update status and logs columns
            # Assuming columns are: A=image_url, B=image_metadata, C=video_output, 
            #                       D=status, E=logs
            range_name = f"{sheet_title}!D{row_num}:E{row_num}"
            
            values = [[status, logs]]
            
            body = {
                'values': values
            }
            
            self.sheets_service.spreadsheets().values().update(
                spreadsheetId=sheet_id,
                range=range_name,
                valueInputOption='RAW',
                body=body
            ).execute()
            
            logger.info(f"Updated sheet row {row_num}: status={status}")
            
        except Exception as e:
            logger.warning(f"Could not update sheet row {row_idx}: {e}")
            # Don't raise - sheet update is optional
    
    def process_sheet_to_bulk(self, sheet_url: str) -> tuple[str, dict]:
        """
        Main Phase 1 function: Process Google Sheet and upload to GCS
        
        Returns:
            - bulk_id: Unique identifier for this bulk operation
            - status_summary: Dict with processing status counts
        """
        status_summary = {
            "READY_TO_GENERATE": 0,
            "SKIPPED": 0,
            "ERROR": 0
        }
        
        try:
            logger.info("Starting Phase 1: Sheet to Bulk")
            
            # Get sheet data
            sheet_data = sheets_svc.get_sheet_data(sheet_url)
            
            if not sheet_data:
                raise ValueError("No data found in sheet")
            
            # Extract spreadsheet ID and sheet title from URL
            sheet_id = sheets_svc.extract_sheet_id_from_url(sheet_url)
            
            # Get sheet title
            spreadsheet = self.sheets_service.spreadsheets().get(
                spreadsheetId=sheet_id
            ).execute()
            sheet_title = spreadsheet['sheets'][0]['properties']['title']
            
            # Generate bulk_id
            timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
            bulk_id = f"{timestamp}_{sheet_id}"
            logger.info(f"Generated bulk_id: {bulk_id}")
            
            # Process each row
            for row_idx, row_data in enumerate(sheet_data, start=1):
                row_id = f"row_{row_idx}"
                row_status = "ERROR"
                row_logs = ""
                
                try:
                    # Get image URL
                    image_url = row_data.get('image_url', '').strip()
                    
                    if not image_url:
                        if SKIP_ROWS_WITHOUT_IMAGE:
                            logger.warning(f"{row_id}: No image_url provided")
                            row_status = "SKIPPED"
                            row_logs = "No image_url provided"
                            status_summary["SKIPPED"] += 1
                            self._update_sheet_row_status(sheet_id, sheet_title, row_idx, 
                                                         row_status, row_logs)
                            continue
                        else:
                            raise ValueError("image_url is required")
                    
                    # Download image
                    row_logs = "Downloading image..."
                    self._update_sheet_row_status(sheet_id, sheet_title, row_idx, 
                                                 "PROCESSING", row_logs)
                    image_data = self._download_image_from_url(image_url)
                    filename = get_file_name_from_uri(image_url).split('?')[0]
                    
                    # Build GCS paths (using RELATIVE paths internally)
                    base_path = get_bulk_bucket_base_path(bulk_id)
                    row_image_path = get_bulk_row_image_bucket_base_path(bulk_id, row_id)
                    
                    # Upload image
                    row_logs = f"Uploading image to GCS: {filename}"
                    self._update_sheet_row_status(sheet_id, sheet_title, row_idx, 
                                                 "PROCESSING", row_logs)
                    image_full_path = f"{row_image_path}{filename}"
                    self._upload_image_to_gcs(image_full_path, image_data, filename)
                    
                    # Upload image metadata
                    row_logs = "Uploading image metadata..."
                    self._update_sheet_row_status(sheet_id, sheet_title, row_idx, 
                                                 "PROCESSING", row_logs)
                    image_metadata = {
                        "original_url": image_url,
                        "uploaded_at": datetime.datetime.now().isoformat(),
                        "filename": filename
                    }
                    image_metadata_path = f"{row_image_path}image_metadata.json"
                    self._upload_metadata_to_gcs(image_metadata_path, image_metadata)
                    
                    # Upload row metadata
                    row_logs = "Creating row metadata..."
                    self._update_sheet_row_status(sheet_id, sheet_title, row_idx, 
                                                 "PROCESSING", row_logs)
                    row_metadata = {
                        "row_id": row_id,
                        "status": "READY_TO_GENERATE",
                        "image_url": image_url,
                        "image_metadata": row_data.get('image_metadata', {}),
                        "video_output": row_data.get('video_output', ''),
                        "created_at": datetime.datetime.now().isoformat()
                    }
                    row_metadata_path = f"{base_path}{row_id}/metadata.json"
                    self._upload_metadata_to_gcs(row_metadata_path, row_metadata)
                    
                    row_status = "READY_TO_GENERATE"
                    row_logs = f"✓ Uploaded to GCS: {row_image_path}"
                    logger.info(f"✓ {row_id} uploaded successfully")
                    status_summary["READY_TO_GENERATE"] += 1
                    
                except Exception as e:
                    row_status = "ERROR"
                    row_logs = f"Error: {str(e)[:100]}"
                    logger.error(f"Error processing {row_id}: {e}")
                    status_summary["ERROR"] += 1
                    if not CONTINUE_ON_ERROR:
                        self._update_sheet_row_status(sheet_id, sheet_title, row_idx, 
                                                     row_status, row_logs)
                        raise
                
                finally:
                    # Always update sheet with final status
                    self._update_sheet_row_status(sheet_id, sheet_title, row_idx, 
                                                 row_status, row_logs)
            
            logger.info(f"Phase 1 complete. Status: {status_summary}")
            return bulk_id, status_summary
            
        except Exception as e:
            logger.error(f"Phase 1 failed: {e}")
            raise

# Initialize Sheet to Bulk Handler
sheet_to_bulk_handler = SheetToBulkHandler(
    storage_client, sheets_service, GCS_BUCKET
)
print("✓ SheetToBulkHandler initialized")


In [ ]:
# ============================================================================
# EXECUTE PHASE 1: Process Sheet and Upload to GCS
# ============================================================================

print("=" * 80)
print("STARTING PHASE 1: Sheet Upload & GCS Organization")
print("=" * 80)

try:
    BULK_ID_FROM_PHASE_1, PHASE_1_STATUS_SUMMARY = \
        sheet_to_bulk_handler.process_sheet_to_bulk(SHEET_URL)
    
    PHASE_1_COMPLETE = True
    
    print("\n" + "=" * 80)
    print("PHASE 1 EXECUTION SUCCESSFUL ✓")
    print("=" * 80)
    print(f"\nBulk ID: {BULK_ID_FROM_PHASE_1}")
    print(f"Status Summary: {PHASE_1_STATUS_SUMMARY}")
    print(f"\nGCS Base Path: {get_bulk_bucket_base_path(BULK_ID_FROM_PHASE_1)}")
    print("\n→ Ready to proceed to PHASE 2: Video Generation")
    print("→ Scroll down to execute Phase 2\n")
    
except Exception as e:
    PHASE_1_COMPLETE = False
    print("\n" + "=" * 80)
    print(f"PHASE 1 FAILED ✗")
    print("=" * 80)
    print(f"\nError: {e}")
    print("\nPlease check your configuration and try again:")
    print(f"  - SHEET_URL: {SHEET_URL}")
    print(f"  - GCS_BUCKET: {GCS_BUCKET}")
    print(f"  - Required columns: image_url, image_metadata, video_output, status, logs")
    raise

## Section 4: PHASE 2 - Video Generation from Bulk Data

This phase:
1. Reads rows from GCS bulk structure
2. Generates videos using Vertex AI Veo API
3. Tracks video generation status and results
4. Aggregates results across all rows

⚠️ **Prerequisites:** Phase 1 must complete successfully before running Phase 2

In [ ]:
# ============================================================================
# DEBUG: Inspect GCS Structure (Run this to debug seed image discovery)
# ============================================================================

def inspect_gcs_structure(bulk_id):
    """
    Debug utility: Inspect what's actually in GCS for a bulk operation.
    Run this if Phase 2 can't find seed images.
    """
    print(f"\n{'='*80}")
    print(f"DEBUGGING GCS STRUCTURE FOR BULK_ID: {bulk_id}")
    print(f"{'='*80}\n")
    
    base_path = get_bulk_bucket_base_path(bulk_id)
    print(f"Base path: {base_path}\n")
    
    # Parse GCS path
    _, path = base_path.split("gs://", 1)
    bucket_name_part, prefix = path.split("/", 1)
    
    print(f"Bucket: {GCS_BUCKET}")
    print(f"Prefix: {prefix}\n")
    
    # List all files under this bulk_id
    print(f"📁 Files in GCS under {bulk_id}:")
    print("-" * 80)
    
    all_blobs = list(storage_client.list_blobs(GCS_BUCKET, prefix=prefix))
    print(f"Total files found: {len(all_blobs)}\n")
    
    # Group by row
    rows_data = {}
    for blob in all_blobs:
        # Extract row_id
        relative_path = blob.name.replace(prefix, "")
        parts = relative_path.strip("/").split("/")
        
        if parts and parts[0].startswith("row_"):
            row_id = parts[0]
            if row_id not in rows_data:
                rows_data[row_id] = {"metadata": False, "images": [], "videos": []}
            
            # Categorize file
            if "metadata.json" in blob.name:
                rows_data[row_id]["metadata"] = True
            elif "/images_seeds/" in blob.name:
                rows_data[row_id]["images"].append(blob.name.split("/")[-1])
            elif "/video_output/" in blob.name:
                rows_data[row_id]["videos"].append(blob.name.split("/")[-1])
    
    # Print organized view
    for row_id in sorted(rows_data.keys()):
        row_info = rows_data[row_id]
        print(f"📦 {row_id}:")
        print(f"   ✓ Metadata: {row_info['metadata']}")
        print(f"   📷 Images: {row_info['images'] if row_info['images'] else 'NONE FOUND'}")
        print(f"   🎬 Videos: {row_info['videos'] if row_info['videos'] else 'None yet'}")
        print()
    
    return rows_data

# Run debug inspection if PHASE_1_COMPLETE
if PHASE_1_COMPLETE and BULK_ID_FROM_PHASE_1:
    rows_data = inspect_gcs_structure(BULK_ID_FROM_PHASE_1)
else:
    print("⚠️  Phase 1 not completed yet. Run Phase 1 first to inspect GCS structure.")

In [ ]:
# ============================================================================
# DETAILED BLOB INSPECTION (Run if debug shows missing images)
# ============================================================================

def detailed_blob_inspection(bulk_id):
    """
    Very detailed inspection of what's in each row directory.
    """
    print(f"\n{'='*80}")
    print(f"DETAILED BLOB INSPECTION FOR: {bulk_id}")
    print(f"{'='*80}\n")
    
    base_path = get_bulk_bucket_base_path(bulk_id)
    _, path = base_path.split("gs://", 1)
    bucket_name_part, prefix = path.split("/", 1)
    
    print(f"Base prefix: {prefix}\n")
    
    # Get ALL blobs
    all_blobs = list(storage_client.list_blobs(GCS_BUCKET, prefix=prefix))
    print(f"Total blobs: {len(all_blobs)}\n")
    
    print("📋 ALL BLOB PATHS:")
    print("-" * 80)
    for blob in all_blobs:
        print(f"  {blob.name}")
    
    print("\n\n🔍 CHECKING EACH ROW:")
    print("-" * 80)
    
    # Check each row
    for i in range(1, 10):  # Check rows 1-9
        row_id = f"row_{i}"
        print(f"\n{row_id}:")
        
        # Check for metadata
        metadata_path = f"{prefix}{row_id}/metadata.json"
        has_metadata = any(b.name == metadata_path for b in all_blobs)
        print(f"  metadata.json: {has_metadata}")
        
        # Check for images_seeds
        images_prefix = f"{prefix}{row_id}/images_seeds/"
        image_blobs = [b for b in all_blobs if b.name.startswith(images_prefix)]
        if image_blobs:
            print(f"  images_seeds/:")
            for blob in image_blobs:
                filename = blob.name.replace(images_prefix, "")
                print(f"    - {filename}")
        else:
            print(f"  images_seeds/: EMPTY OR NOT FOUND")
        
        # Check for video_output
        video_prefix = f"{prefix}{row_id}/video_output/"
        video_blobs = [b for b in all_blobs if b.name.startswith(video_prefix)]
        if video_blobs:
            print(f"  video_output/: {len(video_blobs)} files")
        
        # Stop if we haven't found this row
        if not has_metadata and not image_blobs:
            break

if PHASE_1_COMPLETE and BULK_ID_FROM_PHASE_1:
    detailed_blob_inspection(BULK_ID_FROM_PHASE_1)
else:
    print("⚠️  Phase 1 not completed yet.")

In [ ]:
# ============================================================================
# PHASE 2: Bulk Video Generator Class Definition
# ============================================================================

class BulkVideoGenerator:
    """
    Generates videos in bulk using Vertex AI Veo API.
    Handles video generation, metadata management, and Google Sheet updates.
    """
    
    def __init__(self, storage_client, bucket_name, project_id, location, model_name):
        """Initialize the bulk video generator"""
        self.storage_client = storage_client
        self.bucket = storage_client.bucket(bucket_name)
        self.bucket_name = bucket_name
        self.project_id = project_id
        self.location = location
        self.model_name = model_name
        logger.info(f"BulkVideoGenerator initialized with model: {model_name}")
    
    def generate_bulk_videos(self, bulk_id: str) -> tuple[list, dict]:
        """
        Generate videos for all rows in a bulk operation.
        
        Args:
            bulk_id: The bulk ID from Phase 1
        
        Returns:
            Tuple of (video_responses list, status_summary dict)
        """
        logger.info(f"Starting bulk video generation for bulk_id: {bulk_id}")
        
        video_responses = []
        status_summary = {
            "VIDEO_GENERATED": 0,
            "ERROR": 0,
            "SKIPPED": 0
        }
        
        try:
            # List all rows in the bulk directory
            base_path = get_bulk_bucket_base_path(bulk_id)
            logger.info(f"Scanning bulk directory: {base_path}")
            
            # Parse GCS path to get the prefix
            _, path = base_path.split("gs://", 1)
            bucket_name_part, prefix = path.split("/", 1)
            
            # List all metadata.json files to find rows
            # Each row has: {base_path}{row_id}/metadata.json
            unique_rows = set()
            for blob in self.storage_client.list_blobs(self.bucket_name, prefix=prefix):
                if blob.name.endswith("/metadata.json"):
                    # Extract row_id from path like: bulk-video-creator/bulk_id/row_1/metadata.json
                    parts = blob.name.split("/")
                    # Find the part that starts with "row_"
                    for part in parts:
                        if part.startswith("row_"):
                            unique_rows.add(part)
                            logger.info(f"Found row: {part}")
                            break
            
            unique_rows = sorted(list(unique_rows))
            logger.info(f"Total rows to process: {len(unique_rows)}")
            
            # Process each row
            for row_id in sorted(unique_rows):
                try:
                    logger.info(f"Processing row: {row_id}")
                    
                    # Get metadata for this row
                    metadata = self._get_row_metadata(bulk_id, row_id)
                    if not metadata:
                        logger.warning(f"No metadata found for {row_id}")
                        status_summary["SKIPPED"] += 1
                        continue
                    
                    # Get seed image
                    seed_image_uri = self._get_seed_image_uri(bulk_id, row_id)
                    if not seed_image_uri:
                        logger.warning(f"No seed image found for {row_id}")
                        status_summary["ERROR"] += 1
                        continue
                    
                    # Generate video
                    response = self._generate_single_video(
                        bulk_id, row_id, seed_image_uri, VIDEO_GENERATION_PROMPT
                    )
                    
                    video_responses.append(response)
                    
                    # Update status
                    if response.get('status') == 'VIDEO_GENERATED':
                        status_summary["VIDEO_GENERATED"] += 1
                    else:
                        status_summary["ERROR"] += 1
                        
                except Exception as e:
                    logger.error(f"Error processing {row_id}: {e}")
                    video_responses.append({
                        'row_id': row_id,
                        'status': 'ERROR',
                        'error': str(e),
                        'logs': f"Error: {str(e)}"
                    })
                    status_summary["ERROR"] += 1
            
            logger.info(f"Bulk video generation complete. Summary: {status_summary}")
            return video_responses, status_summary
            
        except Exception as e:
            logger.error(f"Bulk video generation failed: {e}")
            raise
    
    def _get_row_metadata(self, bulk_id: str, row_id: str) -> dict:
        """Retrieve metadata for a specific row"""
        try:
            base_path = get_bulk_bucket_base_path(bulk_id)
            metadata_path = f"{base_path}{row_id}/metadata.json"
            blob_name = metadata_path.replace(f"gs://{self.bucket_name}/", "")
            
            blob = self.bucket.blob(blob_name)
            if blob.exists():
                metadata_json = blob.download_as_string().decode('utf-8')
                return json.loads(metadata_json)
        except Exception as e:
            logger.warning(f"Could not retrieve metadata for {row_id}: {e}")
        
        return None
    
    def _get_seed_image_uri(self, bulk_id: str, row_id: str) -> str:
        """Get the seed image URI for a row"""
        try:
            images_path = get_bulk_row_image_bucket_base_path(bulk_id, row_id)
            logger.info(f"Looking for seed image in: {images_path}")
            
            # Extract the prefix (remove gs://bucket/ part)
            prefix = images_path.replace(f"gs://{self.bucket_name}/", "")
            logger.info(f"Using prefix: {prefix}")
            
            # List blobs in the images directory
            found_blobs = []
            for blob in self.storage_client.list_blobs(self.bucket_name, prefix=prefix):
                logger.info(f"Found blob: {blob.name}")
                found_blobs.append(blob.name)
                
                # Return the first image file found (skip metadata.json)
                # Images may have extensions or not - accept any file except metadata
                filename = blob.name.split("/")[-1]
                if filename != "image_metadata.json" and filename != "metadata.json":
                    seed_uri = f"gs://{self.bucket_name}/{blob.name}"
                    logger.info(f"Found seed image: {seed_uri}")
                    return seed_uri
            
            if not found_blobs:
                logger.warning(f"No files found at prefix: {prefix}")
            else:
                logger.warning(f"No image files found. Files present: {found_blobs}")
                
        except Exception as e:
            logger.error(f"Error getting seed image for {row_id}: {e}")
            import traceback
            logger.error(traceback.format_exc())
        
        return None
    
    def _generate_single_video(self, bulk_id: str, row_id: str, 
                               seed_image_uri: str, prompt: str) -> dict:
        """
        Generate a single video using Vertex AI Veo REST API.
        
        Returns:
            Response dict with keys: row_id, status, output_path, logs, error
        """
        try:
            logger.info(f"Generating video for {row_id}")
            
            # Download seed image
            seed_image_bytes = self._download_image_from_gcs(seed_image_uri)
            
            # Encode image as base64 for API
            import base64
            image_base64 = base64.standard_b64encode(seed_image_bytes).decode('utf-8')
            
            # Get credentials for API request
            creds, _ = google.auth.default(
                scopes=['https://www.googleapis.com/auth/cloud-platform']
            )
            
            # Create REST API request to Veo model
            logger.info(f"Calling Vertex AI Veo REST API for {row_id}")
            
            # Veo API endpoint
            api_url = (
                f"https://us-central1-aiplatform.googleapis.com/v1/projects/"
                f"{self.project_id}/locations/{self.location}/publishers/google/"
                f"models/{self.model_name}:generateContent"
            )
            
            # Prepare request body for Veo
            # Note: Veo may have specific requirements for video generation
            request_body = {
                "contents": [
                    {
                        "role": "user",
                        "parts": [
                            {
                                "inline_data": {
                                    "mime_type": "image/jpeg",
                                    "data": image_base64
                                }
                            },
                            {
                                "text": prompt
                            }
                        ]
                    }
                ],
                "generation_config": {
                    "max_output_tokens": 2048,
                    "temperature": 0.7
                }
            }
            
            # Make API request with proper authentication
            from google.auth.transport.requests import Request as AuthRequest
            creds.refresh(AuthRequest())
            
            headers = {
                "Authorization": f"Bearer {creds.token}",
                "Content-Type": "application/json"
            }
            
            logger.info(f"Calling Veo endpoint: {api_url}")
            logger.info(f"Request body keys: {request_body.keys()}")
            
            response = requests.post(api_url, json=request_body, headers=headers, timeout=600)
            
            # Check response status and log details
            if response.status_code != 200:
                error_msg = f"API returned {response.status_code}: {response.text}"
                logger.error(f"Veo API Error for {row_id}: {error_msg}")
                raise ValueError(error_msg)
            
            response_json = response.json()
            logger.info(f"Veo API response received for {row_id}")
            
            # Extract video from response
            # Veo returns video in candidates[0].content.parts[0].inline_data
            if not response_json.get('candidates'):
                raise ValueError(f"No candidates in Veo response: {response_json}")
            
            candidate = response_json['candidates'][0]
            if not candidate.get('content', {}).get('parts'):
                raise ValueError(f"No parts in Veo response: {response_json}")
            
            video_part = candidate['content']['parts'][0]
            if 'inline_data' not in video_part:
                raise ValueError(f"No inline_data in video part: {response_json}")
            
            video_base64 = video_part['inline_data'].get('data')
            if not video_base64:
                raise ValueError("No video data in response")
            
            # Decode video from base64
            video_bytes = base64.standard_b64decode(video_base64)
            
            # Upload video to GCS
            output_path = get_bulk_row_video_output_base_path(bulk_id, row_id)
            video_uri = self._upload_video_data_to_gcs(video_bytes, output_path)
            
            logger.info(f"Video generated and uploaded for {row_id} ({len(video_bytes)} bytes)")
            
            return {
                'row_id': row_id,
                'status': 'VIDEO_GENERATED',
                'output_path': video_uri,
                'logs': f"✓ Video generated successfully ({len(video_bytes)} bytes)",
                'video_size': len(video_bytes)
            }
            
        except Exception as e:
            logger.error(f"Video generation failed for {row_id}: {e}")
            import traceback
            logger.error(traceback.format_exc())
            return {
                'row_id': row_id,
                'status': 'ERROR',
                'error': str(e),
                'logs': f"Error during generation: {str(e)[:200]}"
            }
    
    def _download_image_from_gcs(self, gcs_uri: str) -> bytes:
        """Download image bytes from GCS"""
        try:
            _, path = gcs_uri.split("gs://", 1)
            bucket_name, blob_name = path.split("/", 1)
            
            bucket = self.storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            
            return blob.download_as_bytes()
        except Exception as e:
            logger.error(f"Failed to download image from {gcs_uri}: {e}")
            raise
    
    def _upload_video_to_gcs(self, video, output_path: str) -> str:
        """Upload generated video to GCS and return GCS URI"""
        try:
            # Get video bytes
            video_bytes = video.data if hasattr(video, 'data') else video.file.read()
            
            # Upload to GCS
            blob_name = output_path.replace(f"gs://{self.bucket_name}/", "") + "output.mp4"
            blob = self.bucket.blob(blob_name)
            blob.upload_from_string(video_bytes, content_type='video/mp4')
            
            video_uri = f"gs://{self.bucket_name}/{blob_name}"
            logger.info(f"Video uploaded to {video_uri}")
            return video_uri
            
        except Exception as e:
            logger.error(f"Failed to upload video: {e}")
            raise
    
    def _upload_video_data_to_gcs(self, video_bytes: bytes, output_path: str) -> str:
        """Upload video bytes to GCS and return GCS URI"""
        try:
            # Upload to GCS
            blob_name = output_path.replace(f"gs://{self.bucket_name}/", "") + "output.mp4"
            blob = self.bucket.blob(blob_name)
            blob.upload_from_string(video_bytes, content_type='video/mp4')
            
            video_uri = f"gs://{self.bucket_name}/{blob_name}"
            logger.info(f"Video uploaded to {video_uri}")
            return video_uri
            
        except Exception as e:
            logger.error(f"Failed to upload video: {e}")
            raise
    
    def _get_public_url_for_gcs_uri(self, gcs_uri: str) -> str:
        """
        Generate a signed/public URL from a GCS URI.
        Valid for 1 hour.
        """
        try:
            if not gcs_uri.startswith("gs://"):
                return gcs_uri
            
            # Parse gs://bucket/path
            _, path = gcs_uri.split("gs://", 1)
            bucket_name, blob_name = path.split("/", 1)
            
            bucket = self.storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            
            # Generate signed URL (valid for 1 hour)
            signed_url = blob.generate_signed_url(
                version="v4",
                expiration=datetime.timedelta(hours=1),
                method="GET"
            )
            logger.info(f"Generated signed URL for {blob_name}")
            return signed_url
        except Exception as e:
            logger.warning(f"Could not generate signed URL for {gcs_uri}: {e}")
            return gcs_uri  # Return original URI if signing fails
    
    def _update_row_metadata_in_gcs(self, bulk_id: str, row_id: str, 
                                     video_status: str, video_uri: str) -> None:
        """
        Update the metadata.json file for a row with video generation results.
        """
        try:
            base_path = get_bulk_bucket_base_path(bulk_id)
            metadata_path = f"{base_path}{row_id}/metadata.json"
            blob_name = metadata_path.replace(f"gs://{self.bucket_name}/", "")
            
            blob = self.bucket.blob(blob_name)
            
            # Read existing metadata
            metadata_json = blob.download_as_string().decode('utf-8')
            metadata = json.loads(metadata_json)
            
            # Update with video results
            metadata['status'] = video_status
            metadata['video_uri'] = video_uri
            metadata['video_generated_at'] = datetime.datetime.now().isoformat()
            
            # Upload updated metadata
            blob.upload_from_string(
                json.dumps(metadata, indent=2),
                content_type='application/json'
            )
            logger.info(f"Updated metadata for {row_id} with video results")
        except Exception as e:
            logger.warning(f"Could not update metadata for {row_id}: {e}")
    
    def process_video_generation_responses(self, video_responses: list, 
                                           bulk_id: str, sheet_url: str) -> dict:
        """
        Process video generation responses and update Google Sheet with results.
        
        Args:
            video_responses: List of video generation response dicts from _generate_single_video
            bulk_id: The bulk_id for this operation
            sheet_url: Google Sheet URL or ID
        
        Returns:
            Status summary dict
        """
        logger.info("Processing video generation responses and updating sheet")
        
        status_summary = {}
        row_updates = []
        
        try:
            sheet_id = sheets_svc.extract_sheet_id_from_url(sheet_url)
            
            for response in video_responses:
                row_id = response.get('row_id')
                status = response.get('status', 'ERROR')
                logs = response.get('logs', '')
                
                # Extract row number from row_id (e.g., 'row_1' -> 1)
                row_number = int(row_id.split('_')[-1])
                row_index = row_number - 1  # Convert to 0-indexed
                
                # Get video URL or handle error
                if status == 'VIDEO_GENERATED':
                    video_uri = response.get('output_path', '')
                    public_url = self._get_public_url_for_gcs_uri(video_uri)
                    
                    # Update GCS metadata with video results
                    self._update_row_metadata_in_gcs(bulk_id, row_id, status, video_uri)
                else:
                    public_url = ''
                    logs = response.get('error', response.get('logs', 'Unknown error'))
                
                # Track status
                status_summary[status] = status_summary.get(status, 0) + 1
                
                # Accumulate row updates: [row_index, status, logs, video_output]
                row_updates.append([row_index, status, logs, public_url])
                
                logger.info(f"Prepared update for {row_id}: {status}")
                
                # Batch update every 5 rows
                if len(row_updates) >= 5:
                    sheets_svc.batch_append_values(sheet_id, row_updates)
                    row_updates = []
            
            # Final batch update for remaining rows
            if row_updates:
                sheets_svc.batch_append_values(sheet_id, row_updates)
            
            logger.info(f"Video response processing complete. Summary: {status_summary}")
            return status_summary
        
        except Exception as e:
            logger.error(f"Error processing video responses: {e}")
            return status_summary

# Initialize Bulk Video Generator
bulk_video_generator = BulkVideoGenerator(
    storage_client, GCS_BUCKET, GCP_PROJECT_ID, LOCATION, VIDEO_MODEL
)
print(f"✓ BulkVideoGenerator initialized ({VIDEO_MODEL})")
print(f"  Parallel workers: {MAX_PARALLEL_WORKERS}")

In [ ]:
# ============================================================================
# EXECUTE PHASE 2: Generate Bulk Videos and Update Google Sheet
# ============================================================================

# Verify Phase 1 completion
if not PHASE_1_COMPLETE or BULK_ID_FROM_PHASE_1 is None:
    raise RuntimeError(
        "Phase 1 must be completed successfully before running Phase 2.\n"
        "Please scroll up and execute Phase 1 first."
    )

print("=" * 80)
print("STARTING PHASE 2: Video Generation from Bulk Data")
print("=" * 80)
print(f"\nUsing bulk_id from Phase 1: {BULK_ID_FROM_PHASE_1}")
print(f"Phase 1 Status Summary: {PHASE_1_STATUS_SUMMARY}\n")

try:
    # 1. Generate videos
    video_responses, phase2_status_summary = \
        bulk_video_generator.generate_bulk_videos(BULK_ID_FROM_PHASE_1)
    
    # 2. Process responses and update Google Sheet
    print("\n" + "=" * 80)
    print("UPDATING GOOGLE SHEET WITH VIDEO RESULTS")
    print("=" * 80)
    
    final_status_summary = \
        bulk_video_generator.process_video_generation_responses(
            video_responses, BULK_ID_FROM_PHASE_1, SHEET_URL
        )
    
    print("\n" + "=" * 80)
    print("PHASE 2 EXECUTION SUCCESSFUL ✓")
    print("=" * 80)
    print(f"\nVideos Generated: {len(video_responses)}")
    print(f"Status Summary: {final_status_summary}")
    print(f"\nGCS Output Path: {get_bulk_bucket_base_path(BULK_ID_FROM_PHASE_1)}")
    print("✓ Google Sheet updated with status, logs, and video_output columns")
    print("\n→ Proceed to Results Validation section for final review")
    
except Exception as e:
    print("\n" + "=" * 80)
    print(f"PHASE 2 FAILED ✗")
    print("=" * 80)
    print(f"\nError: {e}")
    raise

## Section 5: Results Validation and Export

In [ ]:
# ============================================================================
# Results Validation and Summary
# ==========================================================================

def validate_and_summarize_results():
    """Validate bulk operation results and generate summary"""
    
    print("\n" + "=" * 80)
    print("BULK OPERATION RESULTS VALIDATION")
    print("=" * 80)
    
    # Phase 1 Summary
    print("\n📊 PHASE 1: Sheet Upload & GCS Organization")
    print("-" * 80)
    print(f"Bulk ID: {BULK_ID_FROM_PHASE_1}")
    print(f"Status Summary: {PHASE_1_STATUS_SUMMARY}")
    
    total_phase1 = sum(PHASE_1_STATUS_SUMMARY.values())
    ready = PHASE_1_STATUS_SUMMARY.get("READY_TO_GENERATE", 0)
    skipped = PHASE_1_STATUS_SUMMARY.get("SKIPPED", 0)
    errors = PHASE_1_STATUS_SUMMARY.get("ERROR", 0)
    
    print(f"  Total rows: {total_phase1}")
    print(f"  Ready to generate: {ready}")
    print(f"  Skipped: {skipped}")
    print(f"  Errors: {errors}")
    
    # Phase 2 Summary
    if phase2_status_summary:
        print("\n📊 PHASE 2: Video Generation")
        print("-" * 80)
        print(f"Status Summary: {phase2_status_summary}")
        
        total_phase2 = sum(phase2_status_summary.values())
        generated = phase2_status_summary.get("VIDEO_GENERATED", 0)
        errors2 = phase2_status_summary.get("ERROR", 0)
        
        print(f"  Total processed: {total_phase2}")
        print(f"  Successfully generated: {generated}")
        print(f"  Generation errors: {errors2}")
    
    # GCS Output Location
    print("\n📁 GCS OUTPUT LOCATION")
    print("-" * 80)
    gcs_path = get_bulk_bucket_base_path(BULK_ID_FROM_PHASE_1)
    print(f"Base path: {gcs_path}")
    print(f"\nStructure:")
    print(f"  {gcs_path}")
    print(f"  ├── row_1/")
    print(f"  │   ├── metadata.json (row status & data)")
    print(f"  │   ├── images_seeds/")
    print(f"  │   │   ├── image.jpg (seed image)")
    print(f"  │   │   └── image_metadata.json")
    print(f"  │   └── video_output/")
    print(f"  │       ├── output.mp4 (generated video)")
    print(f"  │       └── generation_metadata.json")
    print(f"  ├── row_2/")
    print(f"  │   ...")
    print(f"  └── row_N/")
    
    # Recommendations
    print("\n💡 NEXT STEPS")
    print("-" * 80)
    print("1. ✓ Phase 1 complete: Data organized in GCS")
    print("2. ✓ Phase 2 complete: Videos generated")
    print("3. Access your results in GCS:")
    print(f"   gsutil ls -r {gcs_path}")
    print("4. Download results locally or integrate with your application")
    print("5. Monitor video quality and generation status")
    
    # Summary JSON
    summary = {
        "execution_timestamp": datetime.datetime.now().isoformat(),
        "bulk_id": BULK_ID_FROM_PHASE_1,
        "gcs_path": gcs_path,
        "phase_1": {
            "status": "COMPLETE",
            "summary": PHASE_1_STATUS_SUMMARY
        },
        "phase_2": {
            "status": "COMPLETE",
            "summary": phase2_status_summary,
            "videos_generated": len(video_responses)
        }
    }
    
    return summary

# Execute validation
try:
    execution_summary = validate_and_summarize_results()
    
    print("\n" + "=" * 80)
    print("✓ BULK OPERATION COMPLETED SUCCESSFULLY")
    print("=" * 80)
    
    # Export summary
    print("\n📋 EXECUTION SUMMARY (JSON):")
    print(json.dumps(execution_summary, indent=2))
    
except NameError as e:
    print("Phase 2 not yet executed. Run Phase 2 first.")
except Exception as e:
    print(f"Error during validation: {e}")
    raise

## Workflow Reference & Troubleshooting

### Execution Flow
```
1. Setup (Automatic)
   ├── Install dependencies
   ├── Import libraries
   └── Authenticate with GCP

2. Configuration (Manual)
   ├── Set GCP_PROJECT_ID
   ├── Set GCS_BUCKET
   ├── Set SHEET_URL
   └── Adjust video generation parameters

3. PHASE 1: Sheet Upload & Organization
   ├── Parse Google Sheet
   ├── Validate columns & data
   ├── Download images from URLs
   └── Upload to GCS bulk structure
   └── Generate bulk_id

4. PHASE 2: Video Generation  
   ├── Read organized data from GCS
   ├── Create video generation requests
   ├── Process videos (sequential or parallel)
   └── Track status & results

5. Results Validation
   ├── Summarize execution
   ├── Validate GCS outputs
   └── Export results JSON
```

### Expected Sheet Format

Your Google Sheet must contain these columns:

| image_url | image_metadata | video_output | status | logs |
|-----------|---------------|--------------|--------|------|
| https://example.com/image1.jpg | {"size": "1920x1080"} | output1 | | |
| https://example.com/image2.jpg | {"size": "1920x1080"} | output2 | | |

### GCS Output Structure

```
gs://bucket/bulk-video-creator/{bulk_id}/
├── row_1/
│   ├── metadata.json
│   ├── images_seeds/
│   │   ├── image.jpg
│   │   └── image_metadata.json
│   └── video_output/
│       ├── output.mp4
│       └── generation_metadata.json
├── row_2/
│   ...
└── row_N/
    ...
```

### Configuration Parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `GCP_PROJECT_ID` | Your Google Cloud project ID | my-project-123 |
| `GCS_BUCKET` | Cloud Storage bucket name | my-bucket |
| `SHEET_URL` | Google Sheets URL or ID | https://docs.google.com/spreadsheets/d/ABC123 |
| `VIDEO_GENERATION_PROMPT` | Prompt for video generation | "Animate with subtle motion..." |
| `MAX_PARALLEL_WORKERS` | Max concurrent video generation tasks | 3 |
| `VIDEO_DURATION_SECONDS` | Duration of output videos | 5 |
| `VIDEO_FPS` | Frames per second | 24 |

### Troubleshooting

**Authentication Issues**
- Ensure you have run `gcloud auth application-default login`
- Check that your service account has necessary permissions
- Verify GCP_PROJECT_ID and credentials

**Sheet Access Issues**
- Verify SHEET_URL is correct
- Ensure the sheet is shared or your account has access
- Check that required columns exist: image_url, image_metadata, video_output, status, logs

**GCS Permission Issues**
- Verify your service account has Storage Admin permissions
- Check GCS_BUCKET exists and is in the correct project
- Ensure bucket name is correct (no gs:// prefix)

**Video Generation Failures**
- Check that seed images are valid and accessible
- Verify video generation prompt is appropriate
- Check Vertex AI API is enabled in GCP
- Monitor API quotas in Cloud Console

### Performance Notes

- Phase 1 speed depends on image download sizes
- Phase 2 can process multiple videos in parallel (controlled by MAX_PARALLEL_WORKERS)
- GCS operations are generally fast; network latency is main bottleneck
- For large batches, consider splitting into multiple smaller bulk operations